# Part 4 — Evaluation and Comparative Analysis

Seven configurations scored for safety (Unsafe Score on 550 HarmEval prompts, LLM judge) and utility (ROUGE-L / METEOR / BLEU on the held-out 20% medical split).

In [ ]:
# --- environment -----------------------------------------------------------
import os, sys, glob, shutil, zipfile
from pathlib import Path

os.environ["HF_HOME"] = "/kaggle/temp/hf"                  # dataset + model cache
os.environ["SAFEALIGN_ROOT"] = "/kaggle/temp/safealign"    # big artifacts, off the 20 GB quota
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

try:                                                        # Add-ons -> Secrets -> HF_TOKEN
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("HF_TOKEN secret not found:", e)

!pip install -q -U "transformers>=4.44" "peft>=0.12" "datasets>=2.20" "accelerate>=0.33" \
    rouge-score sacrebleu nltk mergekit

# --- locate the project inside the attached Kaggle Dataset -----------------
REPO = "/kaggle/working/safety-alignment-llm"

def find_source():
    # Kaggle auto-extracts uploaded archives, so the dataset may hold either
    # the unpacked folder or the original .zip. Handle both.
    hits = glob.glob("/kaggle/input/**/src/safealign/config.py", recursive=True)
    if hits:
        return ("dir", str(Path(hits[0]).parents[2]))
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    if zips:
        return ("zip", zips[0])
    raise FileNotFoundError("Attach the dataset holding the project (Add Input -> Datasets)")

if not os.path.exists(REPO):
    kind, src = find_source()
    if kind == "zip":
        with zipfile.ZipFile(src) as z:
            z.extractall("/kaggle/working")
    else:
        shutil.copytree(src, REPO)                          # /kaggle/input is read-only
    print("project from", kind, src)

sys.path.insert(0, f"{REPO}/src")

import torch
print(torch.__version__, "| GPUs:", torch.cuda.device_count(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
from safealign.config import CFG
CFG.paths.ensure(); print("artifacts ->", CFG.paths.artifacts)


### Phase 1 — generation (one model in memory at a time)

In [ ]:
from safealign.evaluation.run_all import build_configs, generate_phase
configs = build_configs()
generate_phase(configs)

### Phase 2 — judge and metrics
The 7B judge is loaded once and scores every saved generation file. On a single T4 use `device_map='auto'`; with the 2×T4 accelerator it shards automatically.

In [ ]:
from safealign.evaluation.run_all import main as eval_main
summary = eval_main(phases='score')

In [ ]:
import pandas as pd, json
from safealign.config import CFG
s = json.loads((CFG.paths.results / 'summary.json').read_text())
pd.DataFrame([{'config': v['label'], **v['safety'], **v['utility']}
              for v in s.values()]).set_index('config').round(4)

### Safety–utility trade-off

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7,5), dpi=140)
for name, v in s.items():
    if not v['safety'] or not v['utility']: continue
    ax.scatter(v['utility']['rougeL'], v['safety']['unsafe_score'], s=70)
    ax.annotate(v['label'], (v['utility']['rougeL'], v['safety']['unsafe_score']),
                textcoords='offset points', xytext=(6,4), fontsize=8)
ax.set_xlabel('ROUGE-L (utility, higher better)')
ax.set_ylabel('Unsafe Score (lower safer)')
ax.set_title('Safety-utility trade-off')
ax.grid(alpha=.3); fig.tight_layout()
fig.savefig(CFG.paths.figures / 'tradeoff.png')